# JR_train_snow 誤差要因分析

現行パイプラインで検証予測を作り、**WMAEへの寄与が大きい誤差が、どの条件で発生しているか**を調べるNotebookです。

## 方針
1. 現行の特徴量生成・時系列split・LightGBMを再現
2. 行ごとのWMAE寄与を計算
3. 誤差上位ケースを確認
4. 実測値の大きさ別に誤差を確認
5. 月・曜日・列車番号などで誤差を比較
6. 既存の気象特徴量と誤差の関係を見る

**このNotebookでは新しい特徴量を追加しません。**
まず「何を外しているのか」を特定し、その結果を次の特徴量設計につなげます。


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path("/home/keiseki/JR_train_snow")
SRC_ROOT = PROJECT_ROOT / "30.src"
CONFIG_DIR = PROJECT_ROOT / "00.config"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from jr_snow.config import build_feature_columns
from jr_snow.data import load_train_test_data
from jr_snow.features import prepare_model_inputs
from jr_snow.modeling import train_lightgbm_model

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


/home/keiseki/JR_train_snow/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 設定・データ読み込み

In [2]:
with open(CONFIG_DIR / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

with open(CONFIG_DIR / "path.yaml", encoding="utf-8") as f:
    path_config = yaml.safe_load(f)

train_path = Path(path_config["INTERIUM_PATH"]["train_data"])
test_path = Path(path_config["INTERIUM_PATH"]["test_data"])

print("train:", train_path)
print("test :", test_path)
print("split_date:", config["split_date"])

df_train, df_test = load_train_test_data(
    train_data_path=train_path,
    test_data_path=test_path,
)

print("train shape:", df_train.shape)
print("test shape :", df_test.shape)


train: /home/keiseki/JR_train_snow/20.Data/train_exsist_snow_flg.pkl
test : /home/keiseki/JR_train_snow/20.Data/test_exist_snow_flg.pkl
split_date: 2016-12-10
train shape: (15339, 1770)
test shape : (2430, 1064)


In [3]:
feature_columns = build_feature_columns(config)

prepared = prepare_model_inputs(
    df_train=df_train.copy(),
    df_test=df_test.copy(),
    feature_columns=feature_columns,
    split_date=config["split_date"],
    target_col="合計",
)

X_train = prepared["X_train"]
X_valid = prepared["X_valid"]
y_train = prepared["y_train"]
y_valid = prepared["y_valid"]
feature_list = prepared["feature_list"]
categorical_cols = prepared["categorical_cols"]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("features:", len(feature_list))
print("categorical:", categorical_cols)
print("feature count by group:", {
    "base": len(feature_columns["base_features"]),
    "weather": len(feature_columns["weather_features"]),
    "snow": len(feature_columns["snow_features"]),
})
print("先頭の特徴量:", feature_list[:20])
print("天候関連特徴量:", [c for c in feature_list if any(k in c for k in ["降雪量", "日射量", "天気期待値", "降雪", "日照", "日射"])][:20])


train: 2016-01-19 00:00:00 ～ 2016-12-09 00:00:00
valid: 2016-12-10 00:00:00 ～ 2016-12-31 00:00:00
train件数: 14368
valid件数: 971
X_train: (14368, 1358)
X_valid: (971, 1358)
features: 1358
categorical: ['列車番号', '金沢着雪ゼロ列車フラグ', '富山_風向_1_00', '富山_風向_2_00', '富山_風向_3_00', '富山_風向_4_00', '富山_風向_5_00', '富山_風向_6_00', '富山_風向_7_00', '富山_風向_8_00', '富山_風向_9_00', '富山_風向_10_00', '富山_風向_11_00', '富山_風向_12_00', '富山_風向_13_00', '富山_風向_14_00', '富山_風向_15_00', '富山_風向_16_00', '富山_風向_17_00', '富山_風向_18_00', '富山_風向_19_00', '富山_風向_20_00', '富山_風向_21_00', '富山_風向_22_00', '富山_風向_23_00', '富山_風向_0_00', '糸魚川_風向_1_00', '糸魚川_風向_2_00', '糸魚川_風向_3_00', '糸魚川_風向_4_00', '糸魚川_風向_5_00', '糸魚川_風向_6_00', '糸魚川_風向_7_00', '糸魚川_風向_8_00', '糸魚川_風向_9_00', '糸魚川_風向_10_00', '糸魚川_風向_11_00', '糸魚川_風向_12_00', '糸魚川_風向_13_00', '糸魚川_風向_14_00', '糸魚川_風向_15_00', '糸魚川_風向_16_00', '糸魚川_風向_17_00', '糸魚川_風向_18_00', '糸魚川_風向_19_00', '糸魚川_風向_20_00', '糸魚川_風向_21_00', '糸魚川_風向_22_00', '糸魚川_風向_23_00', '糸魚川_風向_0_00', '金沢_風向_1_00', '金沢_風向_2_00', '金沢_風向_3_00', '金沢_風向_4_00'

## 2. 現行モデルで検証予測

`config.yaml` の現在の `MODEL_PARAMS` を使用します。


In [4]:
model = train_lightgbm_model(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    categorical_cols=categorical_cols,
    model_params=config["MODEL_PARAMS"].copy(),
    num_boost_round=1000,
    early_stopping_rounds=100,
)

y_pred = np.asarray(model.predict(X_valid), dtype=float)

print("prediction min :", y_pred.min())
print("prediction max :", y_pred.max())
print("prediction mean:", y_pred.mean())


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[43]	training's WMAE: 0.00419769	valid_1's WMAE: 0.000463814
prediction min : -2.5750879749729502e-05
prediction max : 0.00024730506665270367
prediction mean: 1.1441241680335878e-05


## 3. WMAEと行ごとの誤差寄与

現在の評価式

`weight = 1 + 10000 * abs(y_true)`

に合わせて、各行の `weight × absolute error` を計算します。

**今回の分析では、この「WMAE寄与」が最重要です。**


In [5]:
error_df = X_valid.copy().reset_index(drop=True)
y_true_arr = np.asarray(y_valid.reset_index(drop=True), dtype=float)

error_df["実測"] = y_true_arr
error_df["予測"] = y_pred
error_df["絶対誤差"] = np.abs(error_df["実測"] - error_df["予測"])
error_df["符号付き誤差"] = error_df["予測"] - error_df["実測"]
error_df["WMAE重み"] = 1 + 10000 * np.abs(error_df["実測"])
error_df["WMAE寄与"] = error_df["絶対誤差"] * error_df["WMAE重み"]

wmae = error_df["WMAE寄与"].sum() / error_df["WMAE重み"].sum()
print(f"Validation WMAE = {wmae:.8f}")

error_df.sort_values("WMAE寄与", ascending=False).head(30)


Validation WMAE = 0.00046381


,年,月,日,曜日,列車番号,金沢着雪ゼロ列車フラグ,富山_気温_1_00,富山_気温_2_00,富山_気温_3_00,富山_気温_4_00,富山_気温_5_00,富山_気温_6_00,富山_気温_7_00,富山_気温_8_00,富山_気温_9_00,富山_気温_10_00,富山_気温_11_00,富山_気温_12_00,富山_気温_13_00,富山_気温_14_00,富山_気温_15_00,富山_気温_16_00,富山_気温_17_00,富山_気温_18_00,富山_気温_19_00,富山_気温_20_00,富山_気温_21_00,富山_気温_22_00,富山_気温_23_00,富山_気温_0_00,富山_降水量_1_00,富山_降水量_2_00,富山_降水量_3_00,富山_降水量_4_00,富山_降水量_5_00,富山_降水量_6_00,富山_降水量_7_00,富山_降水量_8_00,富山_降水量_9_00,富山_降水量_10_00,富山_降水量_11_00,富山_降水量_12_00,富山_降水量_13_00,富山_降水量_14_00,富山_降水量_15_00,富山_降水量_16_00,富山_降水量_17_00,富山_降水量_18_00,富山_降水量_19_00,富山_降水量_20_00,富山_降水量_21_00,富山_降水量_22_00,富山_降水量_23_00,富山_降水量_0_00,富山_降雪_1_00,富山_降雪_2_00,富山_降雪_3_00,富山_降雪_4_00,富山_降雪_5_00,富山_降雪_6_00,富山_降雪_7_00,富山_降雪_8_00,富山_降雪_9_00,富山_降雪_10_00,富山_降雪_11_00,富山_降雪_12_00,富山_降雪_13_00,富山_降雪_14_00,富山_降雪_15_00,富山_降雪_16_00,富山_降雪_17_00,富山_降雪_18_00,富山_降雪_19_00,富山_降雪_20_00,富山_降雪_21_00,富山_降雪_22_00,富山_降雪_23_00,富山_降雪_0_00,富山_降雪量_1_00,富山_降雪量_2_00,富山_降雪量_3_00,富山_降雪量_4_00,富山_降雪量_5_00,富山_降雪量_6_00,富山_降雪量_7_00,富山_降雪量_8_00,富山_降雪量_9_00,富山_降雪量_10_00,富山_降雪量_11_00,富山_降雪量_12_00,富山_降雪量_13_00,富山_降雪量_14_00,富山_降雪量_15_00,富山_降雪量_16_00,富山_降雪量_17_00,富山_降雪量_18_00,富山_降雪量_19_00,富山_降雪量_20_00,富山_降雪量_21_00,富山_降雪量_22_00,...,富山野々上_積雪深_軌道A__min_12-18,富山野々上_積雪深_軌道A__min_18-24,富山野々上_積雪深_軌道B__mean_0-6,富山野々上_積雪深_軌道B__mean_6-12,富山野々上_積雪深_軌道B__mean_12-18,富山野々上_積雪深_軌道B__mean_18-24,富山野々上_積雪深_軌道B__max_0-6,富山野々上_積雪深_軌道B__max_6-12,富山野々上_積雪深_軌道B__max_12-18,富山野々上_積雪深_軌道B__max_18-24,富山野々上_積雪深_軌道B__min_0-6,富山野々上_積雪深_軌道B__min_6-12,富山野々上_積雪深_軌道B__min_12-18,富山野々上_積雪深_軌道B__min_18-24,富山_気温_℃__1_00_ge_5.0_C,富山_気温_℃__2_00_ge_5.0_C,富山_気温_℃__3_00_ge_5.0_C,富山_気温_℃__4_00_ge_5.0_C,富山_気温_℃__5_00_ge_5.0_C,富山_気温_℃__6_00_ge_5.0_C,富山_気温_℃__7_00_ge_5.0_C,富山_気温_℃__8_00_ge_5.0_C,富山_気温_℃__9_00_ge_5.0_C,富山_気温_℃__10_00_ge_5.0_C,富山_気温_℃__11_00_ge_5.0_C,富山_気温_℃__12_00_ge_5.0_C,富山_気温_℃__13_00_ge_5.0_C,富山_気温_℃__14_00_ge_5.0_C,富山_気温_℃__15_00_ge_5.0_C,富山_気温_℃__16_00_ge_5.0_C,富山_気温_℃__17_00_ge_5.0_C,富山_気温_℃__18_00_ge_5.0_C,富山_気温_℃__19_00_ge_5.0_C,富山_気温_℃__20_00_ge_5.0_C,富山_気温_℃__21_00_ge_5.0_C,富山_気温_℃__22_00_ge_5.0_C,富山_気温_℃__23_00_ge_5.0_C,富山_気温_℃__0_00_ge_5.0_C,糸魚川_気温_℃__1_00_ge_5.0_C,糸魚川_気温_℃__2_00_ge_5.0_C,糸魚川_気温_℃__3_00_ge_5.0_C,糸魚川_気温_℃__4_00_ge_5.0_C,糸魚川_気温_℃__5_00_ge_5.0_C,糸魚川_気温_℃__6_00_ge_5.0_C,糸魚川_気温_℃__7_00_ge_5.0_C,糸魚川_気温_℃__8_00_ge_5.0_C,糸魚川_気温_℃__9_00_ge_5.0_C,糸魚川_気温_℃__10_00_ge_5.0_C,糸魚川_気温_℃__11_00_ge_5.0_C,糸魚川_気温_℃__12_00_ge_5.0_C,糸魚川_気温_℃__13_00_ge_5.0_C,糸魚川_気温_℃__14_00_ge_5.0_C,糸魚川_気温_℃__15_00_ge_5.0_C,糸魚川_気温_℃__16_00_ge_5.0_C,糸魚川_気温_℃__17_00_ge_5.0_C,糸魚川_気温_℃__18_00_ge_5.0_C,糸魚川_気温_℃__19_00_ge_5.0_C,糸魚川_気温_℃__20_00_ge_5.0_C,糸魚川_気温_℃__21_00_ge_5.0_C,糸魚川_気温_℃__22_00_ge_5.0_C,糸魚川_気温_℃__23_00_ge_5.0_C,糸魚川_気温_℃__0_00_ge_5.0_C,金沢_気温_℃__1_00_ge_5.0_C,金沢_気温_℃__2_00_ge_5.0_C,金沢_気温_℃__3_00_ge_5.0_C,金沢_気温_℃__4_00_ge_5.0_C,金沢_気温_℃__5_00_ge_5.0_C,金沢_気温_℃__6_00_ge_5.0_C,金沢_気温_℃__7_00_ge_5.0_C,金沢_気温_℃__8_00_ge_5.0_C,金沢_気温_℃__9_00_ge_5.0_C,金沢_気温_℃__10_00_ge_5.0_C,金沢_気温_℃__11_00_ge_5.0_C,金沢_気温_℃__12_00_ge_5.0_C,金沢_気温_℃__13_00_ge_5.0_C,金沢_気温_℃__14_00_ge_5.0_C,金沢_気温_℃__15_00_ge_5.0_C,金沢_気温_℃__16_00_ge_5.0_C,金沢_気温_℃__17_00_ge_5.0_C,金沢_気温_℃__18_00_ge_5.0_C,金沢_気温_℃__19_00_ge_5.0_C,金沢_気温_℃__20_00_ge_5.0_C,金沢_気温_℃__21_00_ge_5.0_C,金沢_気温_℃__22_00_ge_5.0_C,金沢_気温_℃__23_00_ge_5.0_C,金沢_気温_℃__0_00_ge_5.0_C,金沢_列車通過時_気温_ge_5.0_C,新高岡_列車通過時_気温_ge_5.0_C,富山_列車通過時_気温_ge_5.0_C,黒部宇奈月温泉_列車通過時_気温_ge_5.0_C,糸魚川_列車通過時_気温_ge_5.0_C,上越妙高_列車通過時_気温_ge_5.0_C,飯山_列車通過時_気温_ge_5.0_C,長野_列車通過時_気温_ge_5.0_C,実測,予測,絶対誤差,符号付き誤差,WMAE重み,WMAE寄与
266,2016,12,16,4,9536E,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-22.0,-22.0,-21.805556,-21.333333,-21.972222,-21.916667,-21.0,-21.0,-21.0,-21.0,-22.0,-22.0,-22.0,-22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

## 4. 誤差上位ケース

In [6]:
top_errors = error_df.sort_values("WMAE寄与", ascending=False).head(50).copy()

show_cols = [
    c for c in [
        "年月日", "年", "月", "日", "曜日",
        "列車番号", "金沢着雪ゼロ列車フラグ",
        "実測", "予測", "絶対誤差", "符号付き誤差",
        "WMAE重み", "WMAE寄与",
    ]
    if c in top_errors.columns
]

top_errors[show_cols]


,年,月,日,曜日,列車番号,金沢着雪ゼロ列車フラグ,実測,予測,絶対誤差,符号付き誤差,WMAE重み,WMAE寄与
266,2016,12,16,4,9536E,1.0,0.003791,-0.000009,0.003800,-0.003800,38.90927,0.147858
302,2016,12,16,4,3502E,1.0,0.002940,0.000050,0.002890,-0.002890,30.40367,0.087870
269,2016,12,16,4,554E,1.0,0.002692,-0.000012,0.002704,-0.002704,27.91817,0.075482
288,2016,12,16,4,574E,0.0,0.002555,0.000016,0.002539,-0.002539,26.54972,0.067397
271,2016,12,16,4,3500E,0.0,0.002224,0.000247,0.001977,-0.001977,23.24091,0.045942
303,2016,12,16,4,562E,0.0,0.001534,0.000021,0.001513,-0.001513,16.33896,0.024720
284,2016,12,16,4,578E,0.0,0.001350,0.000021,0.001329,-0.001329,14.49566,0.019259
286,2016,12,16,4,558E,0.0,0.001255,0.000018,0.001237,-0.001237,13.54784,0.016755
277,2016,12,16,4,566E,1.0,0.001195,0.000050,0.001144,-0.001144,12.94695,0.014816
278,2016,12,16,4,590E,0.0,0.000982,0.000016,0.000966,-0.000966,10.82470,0.010457


## 5. 実測値の大きさ別に誤差を見る

In [7]:
bins = [
    -np.inf,
    0,
    0.0001,
    0.0005,
    0.001,
    0.002,
    0.005,
    0.01,
    0.02,
    0.05,
    0.1,
    0.5,
    1,
    2,
    5,
    10,
    20,
    50,
    np.inf
]

labels = [
    "0",
    "0-0.0001",
    "0.0001-0.0005",
    "0.0005-0.001",
    "0.001-0.002",
    "0.002-0.005",
    "0.005-0.01",
    "0.01-0.02",
    "0.02-0.05",
    "0.05-0.1",
    "0.1-0.5",
    "0.5-1",
    "1-2",
    "2-5",
    "5-10",
    "10-20",
    "20-50",
    "50+"
]


error_df["実測区分"] = pd.cut(
    error_df["実測"], bins=bins, labels=labels, right=True
)

target_summary = error_df.groupby("実測区分", observed=False).agg(
    件数=("実測", "size"),
    実測平均=("実測", "mean"),
    予測平均=("予測", "mean"),
    絶対誤差平均=("絶対誤差", "mean"),
    WMAE寄与合計=("WMAE寄与", "sum"),
).reset_index()

target_summary["WMAE寄与割合"] = (
    target_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
)

target_summary


,実測区分,件数,実測平均,予測平均,絶対誤差平均,WMAE寄与合計,WMAE寄与割合
0,0,953,0.000000,0.000011,0.000015,0.014213,0.024868
1,0-0.0001,0,NaN,NaN,NaN,0.000000,0.000000
2,0.0001-0.0005,1,0.000345,-0.000012,0.000357,0.001586,0.002775
3,0.0005-0.001,8,0.000780,0.000019,0.000762,0.055624,0.097326
4,0.001-0.002,4,0.001333,0.000028,0.001306,0.075550,0.132191
5,0.002-0.005,5,0.002840,0.000059,0.002782,0.424550,0.742839
6,0.005-0.01,0,NaN,NaN,NaN,0.000000,0.000000
7,0.01-0.02,0,NaN,NaN,NaN,0.000000,0.000000
8,0.02-0.05,0,NaN,NaN,NaN,0.000000,0.000000
9,0.05-0.1,0,NaN,NaN,NaN,0.000000,0.000000


In [8]:
plt.figure(figsize=(10, 5))
plt.bar(target_summary["実測区分"].astype(str), target_summary["WMAE寄与割合"])
plt.xticks(rotation=45)
plt.ylabel("WMAE contribution ratio")
plt.xlabel("Actual snow amount")
plt.title("WMAE contribution by actual snow amount")
plt.tight_layout()
plt.show()


/tmp/ipykernel_1239/2151715916.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. 過小予測・過大予測

In [9]:
error_df["予測方向"] = np.where(
    error_df["予測"] >= error_df["実測"],
    "過大予測",
    "過小予測",
)

direction_summary = error_df.groupby("予測方向").agg(
    件数=("実測", "size"),
    実測平均=("実測", "mean"),
    絶対誤差平均=("絶対誤差", "mean"),
    WMAE寄与合計=("WMAE寄与", "sum"),
).reset_index()

direction_summary["WMAE寄与割合"] = (
    direction_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
)

direction_summary


,予測方向,件数,実測平均,絶対誤差平均,WMAE寄与合計,WMAE寄与割合
0,過大予測,684,0.000000,0.000018,0.012391,0.02168
1,過小予測,287,0.000091,0.000095,0.559133,0.97832


## 7. 月・曜日による誤差の偏り

In [10]:
def group_error_summary(df, group_col):
    out = df.groupby(group_col, dropna=False).agg(
        件数=("実測", "size"),
        実測平均=("実測", "mean"),
        予測平均=("予測", "mean"),
        絶対誤差平均=("絶対誤差", "mean"),
        WMAE寄与合計=("WMAE寄与", "sum"),
    ).reset_index()
    out["WMAE寄与割合"] = (
        out["WMAE寄与合計"] / df["WMAE寄与"].sum()
    )
    return out.sort_values("WMAE寄与合計", ascending=False)

if "月" in error_df.columns:
    display(group_error_summary(error_df, "月"))

if "曜日" in error_df.columns:
    display(group_error_summary(error_df, "曜日"))


,月,件数,実測平均,予測平均,絶対誤差平均,WMAE寄与合計,WMAE寄与割合
0,12,971,0.000027,0.000011,0.000041,0.571524,1.0


,曜日,件数,実測平均,予測平均,絶対誤差平均,WMAE寄与合計,WMAE寄与割合
4,4,135,0.000194,8.492665e-06,0.000199,0.558622,0.977426
0,0,132,0.000000,3.124943e-05,0.000031,0.004125,0.007217
6,6,132,0.000000,2.419252e-05,0.000026,0.003456,0.006046
5,5,176,0.000000,8.832009e-06,0.000011,0.001946,0.003404
3,3,132,0.000000,1.177024e-05,0.000012,0.001612,0.002821
2,2,132,0.000000,-3.381970e-06,0.000010,0.001383,0.002420
1,1,132,0.000000,-1.294455e-07,0.000003,0.000380,0.000666


## 8. 列車番号による誤差の偏り

列車番号由来特徴量を追加しても全体スコアがほぼ動かなかったため、
**特定列車だけ極端に外しているのか**を確認します。


In [11]:
if "列車番号" in error_df.columns:
    train_summary = error_df.groupby("列車番号", dropna=False).agg(
        件数=("実測", "size"),
        実測平均=("実測", "mean"),
        予測平均=("予測", "mean"),
        絶対誤差平均=("絶対誤差", "mean"),
        WMAE寄与合計=("WMAE寄与", "sum"),
    ).reset_index()

    train_summary["WMAE寄与割合"] = (
        train_summary["WMAE寄与合計"] / error_df["WMAE寄与"].sum()
    )

    display(
        train_summary
        .sort_values("WMAE寄与合計", ascending=False)
        .head(30)
    )


,列車番号,件数,実測平均,予測平均,絶対誤差平均,WMAE寄与合計,WMAE寄与割合
28,9536E,23,0.000165,0.000008,0.000178,0.148147,0.259214
1,3502E,23,0.000128,0.000013,0.000139,0.088166,0.154265
11,554E,44,0.000061,0.000008,0.000074,0.076030,0.133031
21,574E,44,0.000058,0.000011,0.000072,0.068032,0.119037
0,3500E,23,0.000097,0.000073,0.000149,0.047384,0.082908
15,562E,44,0.000035,0.000011,0.000049,0.025363,0.044377
23,578E,44,0.000031,0.000011,0.000045,0.019901,0.034821
13,558E,44,0.000029,0.000011,0.000043,0.017390,0.030427
17,566E,44,0.000027,0.000012,0.000039,0.015376,0.026904
24,590E,44,0.000022,0.000011,0.000036,0.011093,0.019409


## 9. 実測値 vs 予測値

In [12]:
plt.figure(figsize=(7, 7))
plt.scatter(error_df["実測"], error_df["予測"], alpha=0.35, s=15)

max_value = max(error_df["実測"].max(), error_df["予測"].max())
plt.plot([0, max_value], [0, max_value], linestyle="--")

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs predicted")
plt.tight_layout()
plt.show()


/tmp/ipykernel_1239/2394733165.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. 大きな誤差ケースの気象特徴量

新しい特徴量は作らず、**現在モデルへ投入している気象特徴量**について、
誤差上位5%と検証データ全体の平均を比較します。

差が大きい項目があれば、次の特徴量設計候補です。


In [13]:
weather_cols = [
    c for c in error_df.columns
    if any(key in str(c) for key in [
        "気温", "降水量", "降雪", "積雪", "日照", "日射",
        "風速", "風向", "湿度", "露点", "天気", "気圧", "雲量"
    ])
]

numeric_weather_cols = [
    c for c in weather_cols
    if pd.api.types.is_numeric_dtype(error_df[c])
]

top_n = max(10, int(len(error_df) * 0.05))
top_error_df = error_df.nlargest(top_n, "WMAE寄与")

weather_compare = pd.DataFrame({
    "全検証": error_df[numeric_weather_cols].mean(numeric_only=True),
    "誤差上位5%": top_error_df[numeric_weather_cols].mean(numeric_only=True),
})

weather_compare["差"] = (
    weather_compare["誤差上位5%"] - weather_compare["全検証"]
)
weather_compare["差の絶対値"] = weather_compare["差"].abs()

weather_compare.sort_values("差の絶対値", ascending=False).head(30)


,全検証,誤差上位5%,差,差の絶対値
糸魚川_気温_℃__7_00_ge_5.0_C,0.589083,0.020833,-0.568250,0.568250
糸魚川_気温_℃__6_00_ge_5.0_C,0.589083,0.020833,-0.568250,0.568250
糸魚川_気温_℃__5_00_ge_5.0_C,0.589083,0.020833,-0.568250,0.568250
糸魚川_気温_℃__4_00_ge_5.0_C,0.543769,0.020833,-0.522936,0.522936
糸魚川_気温_℃__3_00_ge_5.0_C,0.498455,0.000000,-0.498455,0.498455
金沢_気温_℃__1_00_ge_5.0_C,0.453141,0.020833,-0.432308,0.432308
金沢_気温_℃__5_00_ge_5.0_C,0.453141,0.020833,-0.432308,0.432308
金沢_気温_℃__2_00_ge_5.0_C,0.453141,0.020833,-0.432308,0.432308
金沢_気温_℃__0_00_ge_5.0_C,0.453141,0.020833,-0.432308,0.432308
糸魚川_気温_℃__1_00_ge_5.0_C,0.453141,0.041667,-0.411474,0.411474


## 11. 大きな着雪を外したケース

In [14]:
large_snow_misses = error_df[
    (error_df["実測"] > error_df["実測"].quantile(0.95))
    & (error_df["予測"] < error_df["実測"])
].sort_values("WMAE寄与", ascending=False)

extra_cols = [c for c in numeric_weather_cols if c not in show_cols]

large_snow_misses[show_cols + extra_cols].head(30)


,年,月,日,曜日,列車番号,金沢着雪ゼロ列車フラグ,実測,予測,絶対誤差,符号付き誤差,WMAE重み,WMAE寄与,富山_気温_1_00,富山_気温_2_00,富山_気温_3_00,富山_気温_4_00,富山_気温_5_00,富山_気温_6_00,富山_気温_7_00,富山_気温_8_00,富山_気温_9_00,富山_気温_10_00,富山_気温_11_00,富山_気温_12_00,富山_気温_13_00,富山_気温_14_00,富山_気温_15_00,富山_気温_16_00,富山_気温_17_00,富山_気温_18_00,富山_気温_19_00,富山_気温_20_00,富山_気温_21_00,富山_気温_22_00,富山_気温_23_00,富山_気温_0_00,富山_降水量_1_00,富山_降水量_2_00,富山_降水量_3_00,富山_降水量_4_00,富山_降水量_5_00,富山_降水量_6_00,富山_降水量_7_00,富山_降水量_8_00,富山_降水量_9_00,富山_降水量_10_00,富山_降水量_11_00,富山_降水量_12_00,富山_降水量_13_00,富山_降水量_14_00,富山_降水量_15_00,富山_降水量_16_00,富山_降水量_17_00,富山_降水量_18_00,富山_降水量_19_00,富山_降水量_20_00,富山_降水量_21_00,富山_降水量_22_00,富山_降水量_23_00,富山_降水量_0_00,富山_降雪_1_00,富山_降雪_2_00,富山_降雪_3_00,富山_降雪_4_00,富山_降雪_5_00,富山_降雪_6_00,富山_降雪_7_00,富山_降雪_8_00,富山_降雪_9_00,富山_降雪_10_00,富山_降雪_11_00,富山_降雪_12_00,富山_降雪_13_00,富山_降雪_14_00,富山_降雪_15_00,富山_降雪_16_00,富山_降雪_17_00,富山_降雪_18_00,富山_降雪_19_00,富山_降雪_20_00,富山_降雪_21_00,富山_降雪_22_00,富山_降雪_23_00,富山_降雪_0_00,富山_降雪量_1_00,富山_降雪量_2_00,富山_降雪量_3_00,富山_降雪量_4_00,富山_降雪量_5_00,富山_降雪量_6_00,富山_降雪量_7_00,富山_降雪量_8_00,富山_降雪量_9_00,富山_降雪量_10_00,富山_降雪量_11_00,富山_降雪量_12_00,富山_降雪量_13_00,富山_降雪量_14_00,富山_降雪量_15_00,富山_降雪量_16_00,...,富山野々上_積雪深_軌道A__max_0-6,富山野々上_積雪深_軌道A__max_6-12,富山野々上_積雪深_軌道A__max_12-18,富山野々上_積雪深_軌道A__max_18-24,富山野々上_積雪深_軌道A__min_0-6,富山野々上_積雪深_軌道A__min_6-12,富山野々上_積雪深_軌道A__min_12-18,富山野々上_積雪深_軌道A__min_18-24,富山野々上_積雪深_軌道B__mean_0-6,富山野々上_積雪深_軌道B__mean_6-12,富山野々上_積雪深_軌道B__mean_12-18,富山野々上_積雪深_軌道B__mean_18-24,富山野々上_積雪深_軌道B__max_0-6,富山野々上_積雪深_軌道B__max_6-12,富山野々上_積雪深_軌道B__max_12-18,富山野々上_積雪深_軌道B__max_18-24,富山野々上_積雪深_軌道B__min_0-6,富山野々上_積雪深_軌道B__min_6-12,富山野々上_積雪深_軌道B__min_12-18,富山野々上_積雪深_軌道B__min_18-24,富山_気温_℃__1_00_ge_5.0_C,富山_気温_℃__2_00_ge_5.0_C,富山_気温_℃__3_00_ge_5.0_C,富山_気温_℃__4_00_ge_5.0_C,富山_気温_℃__5_00_ge_5.0_C,富山_気温_℃__6_00_ge_5.0_C,富山_気温_℃__7_00_ge_5.0_C,富山_気温_℃__8_00_ge_5.0_C,富山_気温_℃__9_00_ge_5.0_C,富山_気温_℃__10_00_ge_5.0_C,富山_気温_℃__11_00_ge_5.0_C,富山_気温_℃__12_00_ge_5.0_C,富山_気温_℃__13_00_ge_5.0_C,富山_気温_℃__14_00_ge_5.0_C,富山_気温_℃__15_00_ge_5.0_C,富山_気温_℃__16_00_ge_5.0_C,富山_気温_℃__17_00_ge_5.0_C,富山_気温_℃__18_00_ge_5.0_C,富山_気温_℃__19_00_ge_5.0_C,富山_気温_℃__20_00_ge_5.0_C,富山_気温_℃__21_00_ge_5.0_C,富山_気温_℃__22_00_ge_5.0_C,富山_気温_℃__23_00_ge_5.0_C,富山_気温_℃__0_00_ge_5.0_C,糸魚川_気温_℃__1_00_ge_5.0_C,糸魚川_気温_℃__2_00_ge_5.0_C,糸魚川_気温_℃__3_00_ge_5.0_C,糸魚川_気温_℃__4_00_ge_5.0_C,糸魚川_気温_℃__5_00_ge_5.0_C,糸魚川_気温_℃__6_00_ge_5.0_C,糸魚川_気温_℃__7_00_ge_5.0_C,糸魚川_気温_℃__8_00_ge_5.0_C,糸魚川_気温_℃__9_00_ge_5.0_C,糸魚川_気温_℃__10_00_ge_5.0_C,糸魚川_気温_℃__11_00_ge_5.0_C,糸魚川_気温_℃__12_00_ge_5.0_C,糸魚川_気温_℃__13_00_ge_5.0_C,糸魚川_気温_℃__14_00_ge_5.0_C,糸魚川_気温_℃__15_00_ge_5.0_C,糸魚川_気温_℃__16_00_ge_5.0_C,糸魚川_気温_℃__17_00_ge_5.0_C,糸魚川_気温_℃__18_00_ge_5.0_C,糸魚川_気温_℃__19_00_ge_5.0_C,糸魚川_気温_℃__20_00_ge_5.0_C,糸魚川_気温_℃__21_00_ge_5.0_C,糸魚川_気温_℃__22_00_ge_5.0_C,糸魚川_気温_℃__23_00_ge_5.0_C,糸魚川_気温_℃__0_00_ge_5.0_C,金沢_気温_℃__1_00_ge_5.0_C,金沢_気温_℃__2_00_ge_5.0_C,金沢_気温_℃__3_00_ge_5.0_C,金沢_気温_℃__4_00_ge_5.0_C,金沢_気温_℃__5_00_ge_5.0_C,金沢_気温_℃__6_00_ge_5.0_C,金沢_気温_℃__7_00_ge_5.0_C,金沢_気温_℃__8_00_ge_5.0_C,金沢_気温_℃__9_00_ge_5.0_C,金沢_気温_℃__10_00_ge_5.0_C,金沢_気温_℃__11_00_ge_5.0_C,金沢_気温_℃__12_00_ge_5.0_C,金沢_気温_℃__13_00_ge_5.0_C,金沢_気温_℃__14_00_ge_5.0_C,金沢_気温_℃__15_00_ge_5.0_C,金沢_気温_℃__16_00_ge_5.0_C,金沢_気温_℃__17_00_ge_5.0_C,金沢_気温_℃__18_00_ge_5.0_C,金沢_気温_℃__19_00_ge_5.0_C,金沢_気温_℃__20_00_ge_5.0_C,金沢_気温_℃__21_00_ge_5.0_C,金沢_気温_℃__22_00_ge_5.0_C,金沢_気温_℃__23_00_ge_5.0_C,金沢_気温_℃__0_00_ge_5.0_C,金沢_列車通過時_気温_ge_5.0_C,新高岡_列車通過時_気温_ge_5.0_C,富山_列車通過時_気温_ge_5.0_C,黒部宇奈月温泉_列車通過時_気温_ge_5.0_C,糸魚川_列車通過時_気温_ge_5.0_C,上越妙高_列車通過時_気温_ge_5.0_C,飯山_列車通過時_気温_ge_5.0_C,長野_列車通過時_気温_ge_5.0_C
266,2016,12,16,4,9536E,1.0,0.003791,-0.000009,0.003800,-0.003800,38.90927,0.147858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-21.0,-21.0,-21.0,-21.0,-22.0,-22.0,-22.0,-22.0,-21.8

## 12. 次の分析へ

このNotebookで特に確認するポイント:

- WMAE寄与上位ケースに共通する条件
- 大きな着雪を過小予測しているか
- 特定の月・時刻・列車番号に誤差が集中しているか
- 特定地域の気象条件で誤差が増えているか
- 現在の気象特徴量では説明できないケースがあるか

ここで**具体的な誤差パターンを1つ見つけてから**、そのパターンを特徴量化します。
